In [1]:
import xarray as xr
import os   
import parflow as pf
import plotly.express as px
from parflow.tools.hydrology import calculate_overland_flow_grid, calculate_subsurface_storage, calculate_water_table_depth
import numpy as np
import shutil
import json
import plotly.io as pio

In [ ]:
# outlet_x = 66
# outlet_y = 135
outlet_x = 18
outlet_y = 21


In [2]:
root_dir = "/glade/derecho/scratch/bwest/drought-ensemble"
domain = "wolf2"
ensemble_name = "basic_5_year_tests"
ensemble_member = "pumping_1e-9"
files = json.load(open(f"{root_dir}/domains/{domain}/processed_full_runs/{ensemble_name}/{ensemble_member}/file_locations.json"))
data = xr.open_mfdataset(files, concat_dim="time", combine="nested")
data.info()

/glade/work/bwest/conda-envs/droughts/lib/python3.11/site-packages/xarray/backends/plugins.py:109: RuntimeWarning: Engine 'cfradial1' loading failed:
No module named 'xarray.core.merge'
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
/glade/work/bwest/conda-envs/droughts/lib/python3.11/site-packages/xarray/backends/plugins.py:109: RuntimeWarning: Engine 'furuno' loading failed:
No module named 'xarray.core.merge'
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
/glade/work/bwest/conda-envs/droughts/lib/python3.11/site-packages/xarray/backends/plugins.py:109: RuntimeWarning: Engine 'gamic' loading failed:
No module named 'xarray.core.merge'
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
ERROR 1: PROJ: proj_create_from_database: Open of /glade/work/bwest/conda-envs/droughts/share/proj failed
/glade/work/bwest/conda-envs/droughts/lib/python3.11/site-packages/xarray/backends/plugins.py:109: RuntimeWarning: 

xarray.Dataset {
dimensions:
	time = 113880 ;
	z = 10 ;
	y = 41 ;
	x = 78 ;

variables:
	float64 pressure(time, z, y, x) ;
	float64 saturation(time, z, y, x) ;
	float64 evaptrans(time, z, y, x) ;
	float64 overland_bc_flux(time, y, x) ;
	float64 mask(time, z, y, x) ;
	float64 mannings(time, y, x) ;
	float64 porosity(time, z, y, x) ;
	float64 specific_storage(time, z, y, x) ;
	float64 DZ_Multiplier(time, z, y, x) ;
	float64 slopex(time, y, x) ;
	float64 slopey(time, y, x) ;
	float64 perm_x(time, z, y, x) ;
	float64 perm_y(time, z, y, x) ;
	float64 perm_z(time, z, y, x) ;
	float64 overland_flow(time, y, x) ;
	float64 subsurface_storage(time, z, y, x) ;
	float64 time(time) ;

// global attributes:
}

In [ ]:
pio.templates.default = "plotly_white"

data["overland_flow"] = data["overland_flow"].isel(x=outlet_x, y=outlet_y)

IndexError: Index 135 is out of bounds for axis 1 with size 41

In [ ]:
# data["year"] = data.time 
# fig = px.line(data.overland_flow.isel(time=slice(365*2, None, 24)).isel(x=outlet_x, y=outlet_y)*1/3600)
# # change the x axis to be in years

# # change x and y axis labels
# fig.update_xaxes(title="")
# fig.update_yaxes(title="")
# # remove x and y axis numbers
# fig.update_xaxes(showticklabels=False)
# fig.update_yaxes(showticklabels=False)

# # remove background grid
# fig.update_layout(
#     xaxis=dict(
#         showgrid=False,
#         showline=True,
#         linewidth=2,
#         linecolor='black'
#     ),
#     yaxis=dict(
#         showgrid=False,
#         showline=True,
#         linewidth=2,

#     )
# )
# # set the height of the figure to be 400
# fig.update_layout(height=200, width=900)



In [ ]:
years = range(2, 13)
cumulative_streamflows = []
for year in years:
    try:
        total_streamflow = data.overland_flow.isel(time=slice(8760*(year-1), 8760*(year),100)).sum(dim=["time"])
        cumulative_streamflows.append(float(total_streamflow))
    except Exception as e:
        print(f"Error with year {year}")
        #print the error
        print(e)
        break

# cumulative_streamflows = xr.concat(cumulative_streamflows, dim="year")

# plot the cumulative streamflows


In [ ]:
fig = px.bar(x=years, y=cumulative_streamflows)

fig.update_xaxes(title="Year")
fig.update_yaxes(title="Cumulative Streamflow (m3)")
fig.show()

In [ ]:
cumulative_streamflows

[4972.443202075355,
 4955.927107816342,
 4934.230510676398,
 4913.81489994539,
 4892.964490353357,
 4873.813619219241,
 4825.692039321285,
 4835.246863024071,
 4803.4774589115505,
 4782.918486837134,
 4749.704993784427]

In [ ]:
streamflow_difference_from_previous_year = []
for year in years:
    if year == 0:
        continue
    streamflow_difference_from_previous_year.append(float((cumulative_streamflows[year-1] - cumulative_streamflows[year-2])/cumulative_streamflows[year-2]))

px.bar(x=years[4:11], y=streamflow_difference_from_previous_year[4:11])



IndexError: list index out of range

In [ ]:
root_dir = "/glade/derecho/scratch/bwest/drought-ensemble"
domain = "wolf"
ensemble_name = "ensemble_1"
ensemble_member = "1_year_drought"
files = json.load(open(f"{root_dir}/domains/{domain}/processed_full_runs/{ensemble_name}/{ensemble_member}/file_locations.json"))
data2 = xr.open_mfdataset(files, concat_dim="time", combine="nested")
data2["overland_flow"] = data2["overland_flow"].isel(x=outlet_x, y=outlet_y)

/glade/derecho/scratch/bwest/tmp/ipykernel_57555/3485673291.py:6: FutureWarning:

In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.



IndexError: Index 135 is out of bounds for axis 1 with size 41

In [ ]:
years = range(1, 10)
cumulative_streamflows2 = []
for year in years:
    total_streamflow = data2.overland_flow.isel(time=slice(8760*(year-1), 8760*(year))).sum(dim="time")
    cumulative_streamflows2.append(float(total_streamflow))

# cumulative_streamflows = xr.concat(cumulative_streamflows, dim="year")

# plot the cumulative streamflows
fig = px.bar(x=years, y=cumulative_streamflows2)

In [ ]:
one_year_drought_recovery_streamflows = cumulative_streamflows2[5:10]